# SAVMON Lewa — Exploratory Data Analysis
**Project:** Savanna Monitoring Pilot (SAVMON), Lewa, Kenya  
**Author:** Vincent Langat  

This notebook covers end-to-end EDA for both challenges in the take-home:

| Section | Challenge | Focus |
|---------|-----------|-------|
| 1 | **Bird** | BirdNET predictions + validation — score distributions, completeness, species coverage, calibration readiness |
| 2 | **Vegetation** | ODK survey tables — completeness, join integrity, QA/QC flags, spatial spread, species richness |

All paths are relative to the project root (`NS/`).

---
## 0 · Setup — imports and display config

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# ── display ──────────────────────────────────────────────────────────────────
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

PALETTE = sns.color_palette('tab10')
sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 4)})

print('All imports OK')

---
# PART 1 — Bird Challenge
## 1.1 · Load data

In [ ]:
bird = pd.read_csv('Data Birds/birdnet_predictions.csv', low_memory=False)
val  = pd.read_csv('Data Birds/validation_results.csv')

print(f'BirdNET predictions : {bird.shape[0]:,} rows × {bird.shape[1]} cols')
print(f'Validation results  : {val.shape[0]:,} rows × {val.shape[1]} cols')

## 1.2 · Data completeness

In [ ]:
def completeness_table(df, label):
    """Return a tidy completeness summary for a DataFrame."""
    n = len(df)
    miss = df.isnull().sum()
    pct  = (miss / n * 100).round(2)
    t = pd.DataFrame({'missing': miss, 'pct_missing': pct, 'dtype': df.dtypes})
    t.index.name = 'column'
    print(f'\n=== {label} — completeness (n={n:,}) ===')
    print(t.to_string())
    return t

bird_comp = completeness_table(bird, 'BirdNET predictions')
val_comp  = completeness_table(val,  'Validation results')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, comp, title in zip(axes,
                            [bird_comp, val_comp],
                            ['BirdNET predictions', 'Validation results']):
    pct = comp['pct_missing'].sort_values(ascending=False)
    if pct.max() == 0:
        ax.text(0.5, 0.5, 'No missing values', ha='center', va='center',
                fontsize=13, transform=ax.transAxes)
        ax.set_title(title)
        ax.axis('off')
    else:
        pct[pct > 0].plot(kind='barh', ax=ax, color=PALETTE[1])
        ax.set_xlabel('% missing')
        ax.set_title(title)

plt.suptitle('Bird data — missing-value rates', y=1.01, fontweight='bold')
plt.tight_layout()
plt.show()

## 1.3 · Schema overview — dtypes and sample rows

In [ ]:
print('--- BirdNET predictions dtypes ---')
print(bird.dtypes)
bird.head(5)

In [ ]:
print('--- Validation results dtypes ---')
print(val.dtypes)
val.head(5)

## 1.4 · Coverage — sites, files, species

In [ ]:
n_sites   = bird['folder'].nunique()
n_files   = bird['begin_path'].nunique()
n_species = bird['common_name'].nunique()
n_codes   = bird['species_code'].nunique()

print(f'Sites (folders)  : {n_sites}')
print(f'Unique WAV files : {n_files:,}')
print(f'Unique species   : {n_species}  ({n_codes} species codes)')
print(f'Total detections : {len(bird):,}')

# validated species
val_species = val['commonName'].nunique()
print(f'\nValidated species   : {val_species}')
print(f'Validation records  : {len(val):,}')
print(f'Positive outcomes   : {val["outcome"].sum():,} ({val["outcome"].mean()*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# detections per site
site_counts = bird['folder'].value_counts().sort_values()
site_counts.plot(kind='barh', ax=axes[0], color=PALETTE[0])
axes[0].set_title('Detections per site')
axes[0].set_xlabel('Number of detections')

# detections per species (top 20)
sp_counts = bird['common_name'].value_counts().head(20).sort_values()
sp_counts.plot(kind='barh', ax=axes[1], color=PALETTE[2])
axes[1].set_title('Top-20 species by detection count')
axes[1].set_xlabel('Number of detections')

# validation records per species
val_sp = val['commonName'].value_counts().sort_values()
val_sp.plot(kind='barh', ax=axes[2], color=PALETTE[3])
axes[2].set_title('Validation records per species')
axes[2].set_xlabel('Number of records')

plt.suptitle('Bird challenge — coverage overview', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 1.5 · BirdNET confidence score distribution

In [ ]:
print('Confidence score — summary statistics')
print(bird['confidence'].describe().to_frame().T.to_string())
print(f"\nSkewness : {bird['confidence'].skew():.3f}")
print(f"Kurtosis : {bird['confidence'].kurt():.3f}")

# distribution by bin
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
labels = [f'{bins[i]:.1f}–{bins[i+1]:.1f}' for i in range(len(bins)-1)]
bird['score_bin'] = pd.cut(bird['confidence'], bins=bins, labels=labels, right=True)
print('\nDetections by confidence bin:')
print(bird['score_bin'].value_counts().sort_index().to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# histogram — all predictions
axes[0].hist(bird['confidence'], bins=50, color=PALETTE[0], edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Confidence score')
axes[0].set_ylabel('Count')
axes[0].set_title('All predictions')

# histogram — validation set, split by outcome
for outcome, color, label in [(1, PALETTE[2], 'TP (outcome=1)'), (0, PALETTE[3], 'FP (outcome=0)')]:
    axes[1].hist(val.loc[val['outcome']==outcome, 'confidence'],
                 bins=30, alpha=0.7, color=color, label=label, edgecolor='white')
axes[1].set_xlabel('Confidence score')
axes[1].set_ylabel('Count')
axes[1].set_title('Validation set by outcome')
axes[1].legend()

# CDF of all predictions
sorted_scores = np.sort(bird['confidence'])
cdf = np.arange(1, len(sorted_scores)+1) / len(sorted_scores)
axes[2].plot(sorted_scores, cdf, color=PALETTE[0], lw=1.5)
for thresh in [0.1, 0.25, 0.5, 0.75]:
    pct_below = (bird['confidence'] <= thresh).mean() * 100
    axes[2].axvline(thresh, color='grey', lw=0.8, ls='--')
    axes[2].text(thresh+0.01, 0.05, f'{thresh:.2f}\n({pct_below:.0f}%)',
                 fontsize=7, color='grey')
axes[2].set_xlabel('Confidence score')
axes[2].set_ylabel('Cumulative proportion')
axes[2].set_title('CDF of confidence scores')

plt.suptitle('BirdNET confidence score distributions', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 1.6 · Per-species score distributions

In [ ]:
sp_stats = (bird.groupby('common_name')['confidence']
            .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
            .rename(columns={'count': 'n_detections'})
            .sort_values('n_detections', ascending=False))

print(f'Species in predictions: {len(sp_stats)}')
print('\nTop-15 by detection count:')
sp_stats.head(15)

In [ ]:
# Box plots for top-15 most-detected species
top15 = sp_stats.head(15).index.tolist()
df_top15 = bird[bird['common_name'].isin(top15)].copy()

# sort by median confidence
order = (df_top15.groupby('common_name')['confidence']
         .median().sort_values().index.tolist())

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df_top15, y='common_name', x='confidence',
            order=order, palette='Blues_d', ax=ax, fliersize=2)
ax.axvline(0.1, color='red', lw=1, ls='--', label='BirdNET min-score (0.1)')
ax.set_xlabel('Confidence score')
ax.set_ylabel('')
ax.set_title('Confidence score distribution — top-15 species by detection count')
ax.legend()
plt.tight_layout()
plt.show()

## 1.7 · Validation set — precision by score threshold

In [ ]:
# Which species appear in both datasets?
pred_species = set(bird['common_name'].unique())
val_species_set = set(val['commonName'].unique())
overlap = pred_species & val_species_set

print(f'Species in predictions : {len(pred_species)}')
print(f'Species in validation  : {len(val_species_set)}')
print(f'Overlap                : {len(overlap)}')
print(f'In val but not pred    : {val_species_set - pred_species}')
print(f'\nValidated species : {sorted(overlap)}')

In [ ]:
thresholds = np.arange(0.05, 1.0, 0.05)
rows = []
for t in thresholds:
    subset = val[val['confidence'] >= t]
    if len(subset) == 0:
        continue
    rows.append({
        'threshold': round(t, 2),
        'n_retained': len(subset),
        'precision': subset['outcome'].mean(),
        'recall_of_pos': subset['outcome'].sum() / val['outcome'].sum()
    })
thresh_df = pd.DataFrame(rows)
print(thresh_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(thresh_df['threshold'], thresh_df['precision'],
             marker='o', ms=4, color=PALETTE[0])
axes[0].axhline(0.9, color='red', lw=1, ls='--', label='90% precision')
axes[0].set_xlabel('Score threshold')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision vs threshold')
axes[0].legend()

axes[1].plot(thresh_df['threshold'], thresh_df['recall_of_pos'],
             marker='o', ms=4, color=PALETTE[1])
axes[1].set_xlabel('Score threshold')
axes[1].set_ylabel('Recall (positives retained)')
axes[1].set_title('Recall vs threshold')

axes[2].plot(thresh_df['threshold'], thresh_df['n_retained'],
             marker='o', ms=4, color=PALETTE[2])
axes[2].set_xlabel('Score threshold')
axes[2].set_ylabel('Records retained')
axes[2].set_title('Records retained vs threshold')

plt.suptitle('Validation set — precision / recall trade-off', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 1.8 · Per-species validation summary

In [ ]:
sp_val = (val.groupby('commonName')
          .agg(
              n_validated=('outcome', 'count'),
              n_positive=('outcome', 'sum'),
              precision=('outcome', 'mean'),
              score_min=('confidence', 'min'),
              score_max=('confidence', 'max'),
              score_mean=('confidence', 'mean'),
          )
          .sort_values('precision', ascending=False)
          .round(3))

print('Per-species validation summary (sorted by precision):')
sp_val

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sp_val_sorted = sp_val.sort_values('precision')
colors = [PALETTE[2] if p >= 0.5 else PALETTE[3] for p in sp_val_sorted['precision']]
axes[0].barh(sp_val_sorted.index, sp_val_sorted['precision'], color=colors)
axes[0].axvline(0.5, color='grey', lw=1, ls='--')
axes[0].set_xlabel('Precision (all validated records)')
axes[0].set_title('Per-species precision across full validation range')

# scatter: n_validated vs precision, size = n_positive
sc = axes[1].scatter(sp_val['n_validated'], sp_val['precision'],
                     s=sp_val['n_positive']*4, alpha=0.7,
                     c=sp_val['score_mean'], cmap='RdYlGn',
                     edgecolors='grey', linewidths=0.5)
plt.colorbar(sc, ax=axes[1], label='Mean confidence score')
for sp, row in sp_val.iterrows():
    axes[1].annotate(sp, (row['n_validated'], row['precision']),
                     fontsize=6.5, ha='left', va='bottom')
axes[1].set_xlabel('Number of validated records')
axes[1].set_ylabel('Precision')
axes[1].set_title('Sample size vs precision (bubble ∝ positives)')

plt.suptitle('Per-species validation outcomes', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 1.9 · Calibration readiness — score range in validation vs predictions

In [ ]:
print('BirdNET predictions — score range coverage by species:')
for sp in sorted(val['commonName'].unique()):
    pred_scores = bird.loc[bird['common_name']==sp, 'confidence']
    val_scores  = val.loc[val['commonName']==sp, 'confidence']
    n_pred  = len(pred_scores)
    n_val   = len(val_scores)
    overlap_low  = max(pred_scores.min(), val_scores.min())
    overlap_high = min(pred_scores.max(), val_scores.max())
    print(f'  {sp:<35} pred={n_pred:5,}  val={n_val:3}  '
          f'pred=[{pred_scores.min():.3f},{pred_scores.max():.3f}]  '
          f'val=[{val_scores.min():.3f},{val_scores.max():.3f}]  '
          f'overlap=[{overlap_low:.3f},{overlap_high:.3f}]')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

species_list = sorted(val['commonName'].unique())
for i, sp in enumerate(species_list):
    pred_s = bird.loc[bird['common_name']==sp, 'confidence']
    val_s  = val.loc[val['commonName']==sp, 'confidence']
    ax.plot([pred_s.min(), pred_s.max()], [i, i],
            color=PALETTE[0], lw=6, alpha=0.4, solid_capstyle='round')
    ax.plot([val_s.min(), val_s.max()], [i, i],
            color=PALETTE[1], lw=3, solid_capstyle='round')

ax.set_yticks(range(len(species_list)))
ax.set_yticklabels(species_list, fontsize=8)
ax.set_xlabel('Confidence score')
ax.set_title('Score range: predictions (wide/light) vs validation (narrow/dark)')
from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0],[0], color=PALETTE[0], lw=6, alpha=0.4, label='Predictions range'),
    Line2D([0],[0], color=PALETTE[1], lw=3, label='Validation range'),
], loc='lower right')
plt.tight_layout()
plt.show()

## 1.10 · Temporal and spatial patterns

In [ ]:
# Extract date, hour from begin_path filenames (format: YYYYMMDD_HHMMSS in filename)
import re

def extract_datetime(path):
    m = re.search(r'(\d{8})_(\d{6})', str(path))
    if m:
        return pd.to_datetime(m.group(1) + m.group(2), format='%Y%m%d%H%M%S')
    return pd.NaT

bird['rec_datetime'] = bird['begin_path'].apply(extract_datetime)
bird['rec_date'] = bird['rec_datetime'].dt.date
bird['rec_hour'] = bird['rec_datetime'].dt.hour
bird['rec_month'] = bird['rec_datetime'].dt.month

n_parsed = bird['rec_datetime'].notna().sum()
print(f'Datetime parsed for {n_parsed:,} / {len(bird):,} rows  ({n_parsed/len(bird)*100:.1f}%)')
print(f"Date range: {bird['rec_date'].min()} → {bird['rec_date'].max()}")
print(f"Hours present: {sorted(bird['rec_hour'].dropna().unique().astype(int))}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# detections by hour of day
hourly = bird['rec_hour'].value_counts().sort_index()
hourly.plot(kind='bar', ax=axes[0], color=PALETTE[0], width=0.85)
axes[0].set_xlabel('Hour of day (24h)')
axes[0].set_ylabel('Detections')
axes[0].set_title('Detections by hour of day')
axes[0].tick_params(axis='x', rotation=0)

# detections by date
daily = bird.groupby('rec_date').size()
daily.plot(kind='bar', ax=axes[1], color=PALETTE[1], width=0.85)
axes[1].set_xlabel('Recording date')
axes[1].set_ylabel('Detections')
axes[1].set_title('Detections by recording date')
axes[1].tick_params(axis='x', rotation=45)

# mean confidence by hour
hourly_conf = bird.groupby('rec_hour')['confidence'].mean()
hourly_conf.plot(kind='bar', ax=axes[2], color=PALETTE[4], width=0.85)
axes[2].set_xlabel('Hour of day (24h)')
axes[2].set_ylabel('Mean confidence')
axes[2].set_title('Mean confidence by hour of day')
axes[2].tick_params(axis='x', rotation=0)

plt.suptitle('BirdNET — temporal patterns', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 1.11 · Site × species detection heatmap

In [ ]:
# restrict to top-20 species for readability
top20_sp = bird['common_name'].value_counts().head(20).index
pivot = (bird[bird['common_name'].isin(top20_sp)]
         .groupby(['folder', 'common_name'])
         .size()
         .unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Detection count'})
ax.set_xlabel('Species')
ax.set_ylabel('Site')
ax.set_title('Site × species detection matrix (top-20 species)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

## 1.12 · Detection duration and frequency bands

In [ ]:
bird['detection_duration_s'] = bird['end_time_s'] - bird['begin_time_s']

print('Detection duration (seconds):')
print(bird['detection_duration_s'].describe())
print('\nFrequency range (Hz):')
print(bird[['low_freq_hz', 'high_freq_hz']].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(bird['detection_duration_s'], bins=30, color=PALETTE[0], edgecolor='white')
axes[0].set_xlabel('Duration (s)')
axes[0].set_ylabel('Count')
axes[0].set_title('Detection duration distribution')

axes[1].scatter(bird['low_freq_hz'], bird['high_freq_hz'],
                alpha=0.1, s=5, color=PALETTE[1])
axes[1].set_xlabel('Low frequency (Hz)')
axes[1].set_ylabel('High frequency (Hz)')
axes[1].set_title('Frequency band of detections')

plt.suptitle('Detection duration & frequency', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 1.13 · Bird EDA — key findings summary

| Finding | Value |
|---------|-------|
| Total BirdNET detections | 29,491 |
| Sites | 60+ |
| Species | see above |
| Validation records | 551 |
| Validated species | all in overlap |
| Score distribution | right-skewed; most detections < 0.3 |
| Completeness | **100%** — no missing values in either file |
| Calibration gap | Validation covers a narrow score slice vs full prediction range — extrapolation needed |

> All detections span 3-second windows (begin → end). Frequency bands are consistently 0–15 kHz (BirdNET default). Temporal parsing succeeds for all rows where filename follows the `YYYYMMDD_HHMMSS` convention.

---
# PART 2 — Vegetation Challenge
## 2.1 · Load all tables

In [ ]:
survey   = pd.read_csv('Data Vegetation/ODK Data Exports/herbaceous_veg_survey.csv', low_memory=False)
quadrat  = pd.read_csv('Data Vegetation/ODK Data Exports/herbaceous_veg_survey-quadrat_repeat.csv', low_memory=False)
addsp    = pd.read_csv('Data Vegetation/ODK Data Exports/herbaceous_veg_survey-additional_species_repeat.csv', low_memory=False)
regplots = pd.read_csv('Data Vegetation/ODK Data Exports/register_vegetation_plots.csv', low_memory=False)
vegplots = pd.read_csv('Data Vegetation/Entity lists/vegplots.csv', low_memory=False)
species  = pd.read_csv('Data Vegetation/Entity lists/species.csv', low_memory=False)
sp_extra = pd.read_csv('Data Vegetation/Entity lists/species_extra.csv', low_memory=False)
centroids = pd.read_csv('Data Vegetation/Entity lists/centroids.csv', low_memory=False)

tables = {
    'survey': survey, 'quadrat': quadrat, 'addsp': addsp,
    'regplots': regplots, 'vegplots': vegplots,
    'species': species, 'sp_extra': sp_extra, 'centroids': centroids
}

for name, df in tables.items():
    print(f'{name:<12} {df.shape[0]:>5} rows × {df.shape[1]:>3} cols')

## 2.2 · Data completeness — all vegetation tables

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for ax, (name, df) in zip(axes, tables.items()):
    pct = (df.isnull().mean() * 100).sort_values(ascending=False)
    pct_nonzero = pct[pct > 0]
    if len(pct_nonzero) == 0:
        ax.text(0.5, 0.5, 'No missing values', ha='center', va='center',
                fontsize=11, transform=ax.transAxes)
        ax.set_title(name)
        ax.axis('off')
    else:
        pct_nonzero.head(15).plot(kind='barh', ax=ax, color=PALETTE[1])
        ax.set_xlabel('% missing')
        ax.set_title(f'{name}  (n={len(df)})')
        ax.tick_params(axis='y', labelsize=7)

plt.suptitle('Vegetation tables — missing-value rates (top-15 per table)', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2.3 · Survey table overview

In [ ]:
# Parse timestamps
survey['start_time'] = pd.to_datetime(survey['survey_begin-start_time'], errors='coerce', utc=True)
survey['end_time']   = pd.to_datetime(survey['survey_end-end_time'],     errors='coerce', utc=True)
survey['duration_min'] = (survey['end_time'] - survey['start_time']).dt.total_seconds() / 60

print('Survey table summary')
print(f"  Submissions          : {len(survey)}")
print(f"  Unique plots surveyed: {survey['plot_selection-plot_name'].nunique()}")
print(f"  Recorders            : {survey['field_team_specifics-recorder_label'].nunique()} — {survey['field_team_specifics-recorder_label'].unique().tolist()}")
print(f"  Date range           : {survey['start_time'].min().date()} → {survey['end_time'].max().date()}")
print(f"  Survey duration — mean: {survey['duration_min'].mean():.1f} min, median: {survey['duration_min'].median():.1f} min")
print(f"  Missing GPS          : {survey['survey_end-background_geopoint-Latitude'].isna().sum()} / {len(survey)}")
print(f"  ReviewState values   : {survey['ReviewState'].value_counts().to_dict()}")

In [ ]:
# Duplicate plot submissions (same plot surveyed more than once)
plot_counts = survey['plot_selection-plot_name'].value_counts()
dups = plot_counts[plot_counts > 1]
print(f'Plots surveyed more than once: {len(dups)}')
if len(dups):
    print(dups.to_string())
    dup_rows = survey[survey['plot_selection-plot_name'].isin(dups.index)]
    print('\nDuplicate survey rows:')
    print(dup_rows[['plot_selection-plot_name', 'field_team_specifics-recorder_label',
                    'start_time', 'end_time', 'ReviewState']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Survey durations
axes[0].hist(survey['duration_min'].dropna(), bins=15, color=PALETTE[0], edgecolor='white')
axes[0].set_xlabel('Duration (minutes)')
axes[0].set_ylabel('Count')
axes[0].set_title('Survey durations')

# Submissions by recorder
survey['field_team_specifics-recorder_label'].value_counts().plot(kind='bar', ax=axes[1], color=PALETTE[1])
axes[1].set_xlabel('Recorder')
axes[1].set_ylabel('Surveys')
axes[1].set_title('Surveys per recorder')
axes[1].tick_params(axis='x', rotation=0)

# Quadrat count per survey
axes[2].hist(survey['observations-quadrat_repeat_count'], bins=10, color=PALETTE[2], edgecolor='white')
axes[2].set_xlabel('Number of quadrats')
axes[2].set_ylabel('Surveys')
axes[2].set_title('Quadrat count per survey')

plt.suptitle('Survey table — operational overview', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2.4 · Join integrity — survey → quadrat → addsp

In [ ]:
# survey KEY → quadrat PARENT_KEY → addsp PARENT_KEY (two levels)
survey_keys  = set(survey['KEY'])
quadrat_pkeys = set(quadrat['PARENT_KEY'])
addsp_pkeys   = set(addsp['PARENT_KEY'])

# quadrat parent keys that match survey
q_matched   = quadrat_pkeys & survey_keys
q_orphan    = quadrat_pkeys - survey_keys

# addsp parent keys that match quadrat keys
quadrat_keys = set(quadrat['KEY'])
a_matched   = addsp_pkeys & quadrat_keys
a_orphan    = addsp_pkeys - quadrat_keys

print('Join integrity check')
print(f"  Survey records                : {len(survey):>4}")
print(f"  Quadrat records               : {len(quadrat):>4}")
print(f"  Additional species records    : {len(addsp):>4}")
print()
print(f"  Quadrat PARENT_KEY → survey KEY")
print(f"    Matched  : {len(q_matched)}")
print(f"    Orphaned : {len(q_orphan)} (quadrats with no parent survey)")
print()
print(f"  Addsp PARENT_KEY → quadrat KEY")
print(f"    Matched  : {len(a_matched)}")
print(f"    Orphaned : {len(a_orphan)} (addsp records with no parent quadrat)")

# surveys with no quadrats at all
surveys_with_quadrats = set(quadrat['PARENT_KEY'])
surveys_no_q = survey_keys - surveys_with_quadrats
print(f"\n  Survey records with zero quadrat children: {len(surveys_no_q)}")

In [ ]:
# Quadrats per survey
q_per_survey = quadrat.groupby('PARENT_KEY').size().rename('n_quadrats')
q_per_survey = q_per_survey.reindex(survey['KEY'], fill_value=0)
q_per_survey.index = survey['plot_selection-plot_name'].values

# Addsp per quadrat
a_per_q = addsp.groupby('PARENT_KEY').size().rename('n_addsp')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

q_per_survey.sort_values().plot(kind='barh', ax=axes[0], color=PALETTE[0])
axes[0].axvline(20, color='red', lw=1.5, ls='--', label='Expected (20)')
axes[0].set_xlabel('Quadrat records')
axes[0].set_title('Quadrat records per survey')
axes[0].legend()
axes[0].tick_params(axis='y', labelsize=7)

a_per_q.hist(bins=20, ax=axes[1], color=PALETTE[1], edgecolor='white')
axes[1].set_xlabel('Additional species per quadrat')
axes[1].set_ylabel('Count')
axes[1].set_title('Additional species records per quadrat')

plt.suptitle('Join counts — surveys → quadrats → additional species', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2.5 · Quadrat-level completeness and herb presence

In [ ]:
print('Quadrat table completeness summary')
print(f"  Total quadrat records  : {len(quadrat)}")
print(f"  Herbs present          : {(quadrat['herbs_present']=='yes').sum()} / {quadrat['herbs_present'].notna().sum()} ({(quadrat['herbs_present']=='yes').mean()*100:.1f}%)")
print(f"  All unknown/unid       : {(quadrat['herb_species-all_unknown_unidentifiable']=='yes').sum()}")
print(f"  Additional sp present  : {(quadrat['additional_species_present']=='yes').sum()}")
print(f"  Missing herb species   : {quadrat['herb_species-selected_herb_species_uuids'].isna().sum()}")
print(f"  Species count (known)  : mean={quadrat['herb_species-count_herb_species'].mean():.2f}, max={quadrat['herb_species-count_herb_species'].max():.0f}")
print(f"  GPS recorded           : {quadrat['location_quadrat-Latitude'].notna().sum()} / {len(quadrat)}")
print(f"  Has image              : {quadrat['quadrat_image'].notna().sum()} / {len(quadrat)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# herb presence
hp = quadrat['herbs_present'].value_counts(dropna=False)
hp.plot(kind='bar', ax=axes[0], color=[PALETTE[2], PALETTE[3]])
axes[0].set_xlabel('')
axes[0].set_ylabel('Quadrats')
axes[0].set_title('Herb presence per quadrat')
axes[0].tick_params(axis='x', rotation=0)

# known herb species count per quadrat
quadrat['herb_species-count_herb_species'].dropna().astype(int).value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color=PALETTE[0], width=0.85
)
axes[1].set_xlabel('Known herb species count')
axes[1].set_ylabel('Quadrats')
axes[1].set_title('Known herb species per quadrat')
axes[1].tick_params(axis='x', rotation=0)

# GPS accuracy distribution
quadrat['location_quadrat-Accuracy'].dropna().hist(bins=30, ax=axes[2], color=PALETTE[4], edgecolor='white')
axes[2].set_xlabel('GPS accuracy (m)')
axes[2].set_ylabel('Quadrats')
axes[2].set_title('GPS accuracy at quadrat level')

plt.suptitle('Quadrat records — key field distributions', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2.6 · Species richness — all species per quadrat (known + additional)

In [ ]:
# Parse all_species_list (pipe/HTML-separated) and count
def count_species(cell):
    if pd.isna(cell):
        return 0
    parts = [p.strip() for p in str(cell).replace('<br/>', '|').split('|') if p.strip()]
    return len(parts)

quadrat['n_species_all'] = quadrat['species_list-all_species_list'].apply(count_species)

print('Total species per quadrat (known + additional):')
print(quadrat['n_species_all'].describe())

# Join back to survey to get plot-level richness
q_with_plot = quadrat.merge(
    survey[['KEY', 'plot_selection-plot_name']],
    left_on='PARENT_KEY', right_on='KEY', how='left',
    suffixes=('', '_survey')
)
plot_richness = (q_with_plot.groupby('plot_selection-plot_name')['n_species_all']
                 .agg(['sum', 'mean', 'max'])
                 .rename(columns={'sum': 'total_occ', 'mean': 'mean_per_q', 'max': 'max_per_q'})
                 .sort_values('mean_per_q', ascending=False))
print('\nPlot-level richness (quadrat-averaged):')
print(plot_richness.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# histogram of quadrat richness
quadrat['n_species_all'].hist(bins=20, ax=axes[0], color=PALETTE[0], edgecolor='white')
axes[0].set_xlabel('Total species per quadrat')
axes[0].set_ylabel('Quadrats')
axes[0].set_title('Species richness per quadrat')

# mean species per quadrat by plot
plot_richness['mean_per_q'].sort_values().plot(kind='barh', ax=axes[1], color=PALETTE[2])
axes[1].set_xlabel('Mean species per quadrat')
axes[1].set_title('Mean richness per quadrat by plot')
axes[1].tick_params(axis='y', labelsize=7)

# species richness contribution: known vs additional
known_mean = quadrat['herb_species-count_herb_species'].mean()
addsp_mean_per_q = len(addsp) / len(quadrat)
bars = axes[2].bar(['Known\n(species list)', 'Additional\n(addsp repeat)'],
                   [known_mean, addsp_mean_per_q],
                   color=[PALETTE[0], PALETTE[1]])
axes[2].bar_label(bars, fmt='%.2f')
axes[2].set_ylabel('Mean per quadrat')
axes[2].set_title('Known vs additional species per quadrat')

plt.suptitle('Vegetation — species richness', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2.7 · Additional species — entry modes and review status

In [ ]:
print('Additional species entry modes:')
print(addsp['species_entry_mode'].value_counts().to_string())
print(f'\nTotal addsp records: {len(addsp)}')
print(f'Unique validated names: {addsp["validated_name"].nunique()} (1 missing: {addsp["validated_name"].isna().sum()})')
print(f'\nReview status breakdown:')
print(addsp['review_status'].value_counts(dropna=False).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

addsp['species_entry_mode'].value_counts().plot(kind='bar', ax=axes[0], color=PALETTE[0], width=0.7)
axes[0].set_xlabel('Entry mode')
axes[0].set_ylabel('Records')
axes[0].set_title('Additional species — entry modes')
axes[0].tick_params(axis='x', rotation=20)

rs = addsp['review_status'].value_counts(dropna=False)
rs.index = rs.index.fillna('NaN (not applicable)')
rs.plot(kind='bar', ax=axes[1], color=PALETTE[1], width=0.7)
axes[1].set_xlabel('Review status')
axes[1].set_ylabel('Records')
axes[1].set_title('Additional species — review status')
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('Additional species repeat table', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2.8 · Species entity list — taxonomic coverage

In [ ]:
print('Species entity list overview')
print(f"  Total species  : {len(species)}")
print(f"  Groups         : {species['groups'].value_counts().to_dict()}")
print(f"  Families (top 10):")
print(species['family'].value_counts().head(10).to_string())
print(f"\n  Missing vernacular name: {species['vernacular_name'].isna().sum()} / {len(species)}")
print(f"  Missing class          : {species['class'].isna().sum()} / {len(species)}")

print('\nSpecies extra list overview')
print(f"  Total records  : {len(sp_extra)}")
print(f"  Choice status  : {sp_extra['choice_status'].value_counts().to_dict()}")
print(f"  Group types    : {sp_extra['group_type'].value_counts().to_dict()}")
print(f"  Review status  : {sp_extra['review_status'].value_counts(dropna=False).to_dict()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# species by group
species['groups'].value_counts().plot(kind='bar', ax=axes[0], color=PALETTE[0], width=0.7)
axes[0].set_xlabel('Group')
axes[0].set_ylabel('Species')
axes[0].set_title('Species entity list by group')
axes[0].tick_params(axis='x', rotation=20)

# top-15 families
top_fam = species['family'].value_counts().head(15).sort_values()
top_fam.plot(kind='barh', ax=axes[1], color=PALETTE[1])
axes[1].set_xlabel('Species count')
axes[1].set_title('Top-15 families in species list')
axes[1].tick_params(axis='y', labelsize=8)

# species_extra: group type
sp_extra['group_type'].value_counts().plot(kind='bar', ax=axes[2], color=PALETTE[2], width=0.7)
axes[2].set_xlabel('Group type')
axes[2].set_ylabel('Records')
axes[2].set_title('species_extra — group types')
axes[2].tick_params(axis='x', rotation=0)

plt.suptitle('Species entity lists — taxonomic coverage', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2.9 · Plot registration — ecosystem metadata

In [ ]:
print('Registered plots overview')
print(f"  Total registrations: {len(regplots)}")
print(f"  Unique plots       : {regplots['plot_selection-plot_name'].nunique()}")
print(f"\nEcosystem types:")
print(regplots['plot_metadata-ecosystem_type_name'].value_counts(dropna=False).to_string())
print(f"\nSoil types:")
print(regplots['plot_metadata-soil_type'].value_counts(dropna=False).to_string())
print(f"\nCanopy structure:")
print(regplots['plot_metadata-canopy_structure'].value_counts(dropna=False).to_string())
print(f"\nDisturbance types (top values):")
print(regplots['plot_metadata-disturbance_types'].value_counts(dropna=False).head(10).to_string())
print(f"\nMissing endpoint A GPS: {regplots['plot_metadata-collect_points-endpoint_a-location_endpoint_a-Latitude'].isna().sum()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ecosystem type
regplots['plot_metadata-ecosystem_type_name'].value_counts(dropna=False).plot(
    kind='bar', ax=axes[0], color=PALETTE[0], width=0.7)
axes[0].set_xlabel('')
axes[0].set_ylabel('Plots')
axes[0].set_title('Ecosystem types')
axes[0].tick_params(axis='x', rotation=20)

# canopy height
regplots['plot_metadata-canopy_height_m'].dropna().hist(bins=15, ax=axes[1], color=PALETTE[1], edgecolor='white')
axes[1].set_xlabel('Canopy height (m)')
axes[1].set_ylabel('Plots')
axes[1].set_title('Canopy height distribution')

# ground cover types (split multi-value)
gc_all = (regplots['plot_metadata-ground_cover'].dropna()
          .str.split().explode())
gc_all.value_counts().plot(kind='bar', ax=axes[2], color=PALETTE[3], width=0.8)
axes[2].set_xlabel('Ground cover type')
axes[2].set_ylabel('Count')
axes[2].set_title('Ground cover types (multi-select)')
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('Plot registration — ecosystem metadata', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2.10 · Spatial distribution of plots

In [ ]:
# Parse geometry from vegplots (format: "lat lon alt acc")
def parse_geometry(g):
    try:
        parts = str(g).split()
        return float(parts[0]), float(parts[1])
    except:
        return np.nan, np.nan

vegplots[['lat', 'lon']] = vegplots['geometry'].apply(
    lambda g: pd.Series(parse_geometry(g)))

# Also parse centroids
centroids[['lat', 'lon']] = centroids['geometry'].apply(
    lambda g: pd.Series(parse_geometry(g)))

print(f'vegplots with valid coords: {vegplots["lat"].notna().sum()} / {len(vegplots)}')
print(f'centroids with valid coords: {centroids["lat"].notna().sum()} / {len(centroids)}')
print(f"\nPlot lat range : {vegplots['lat'].min():.4f} → {vegplots['lat'].max():.4f}")
print(f"Plot lon range : {vegplots['lon'].min():.4f} → {vegplots['lon'].max():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot positions: vegplots (surveyed) vs centroids (all planned)
centroid_primary = centroids[centroids['sample_status']=='primary']
centroid_backup  = centroids[centroids['sample_status']=='backup']

axes[0].scatter(centroid_backup['lon'], centroid_backup['lat'],
                s=40, marker='x', color='lightgrey', label='Centroid (backup)', zorder=1)
axes[0].scatter(centroid_primary['lon'], centroid_primary['lat'],
                s=50, marker='s', color=PALETTE[4], alpha=0.6, label='Centroid (primary)', zorder=2)
axes[0].scatter(vegplots['lon'], vegplots['lat'],
                s=80, color=PALETTE[0], label='Registered vegplot', zorder=3, edgecolors='white')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('Plot locations — planned vs registered')
axes[0].legend(fontsize=8)

# GPS accuracy at survey level
survey_gps_acc = survey['survey_end-background_geopoint-Accuracy'].dropna()
axes[1].hist(survey_gps_acc, bins=15, color=PALETTE[2], edgecolor='white')
axes[1].axvline(5, color='red', lw=1.5, ls='--', label='5 m threshold')
axes[1].set_xlabel('GPS accuracy (m)')
axes[1].set_ylabel('Surveys')
axes[1].set_title('Survey-level GPS accuracy')
axes[1].legend()

plt.suptitle('Spatial overview — Lewa study area', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2.11 · QA/QC checks — key flags

In [ ]:
qc = {}

# ── survey-level flags ────────────────────────────────────────────────────────
# 1. Rejected submissions
qc['rejected_submissions'] = (survey['ReviewState'] == 'rejected').sum()

# 2. Missing GPS at survey end
qc['missing_survey_gps'] = survey['survey_end-background_geopoint-Latitude'].isna().sum()

# 3. Duplicate plot surveys
qc['duplicate_plot_surveys'] = (survey['plot_selection-plot_name'].value_counts() > 1).sum()

# 4. Survey duration anomalies (< 20 min or > 180 min)
qc['short_surveys_lt20min'] = (survey['duration_min'] < 20).sum()
qc['long_surveys_gt180min'] = (survey['duration_min'] > 180).sum()

# 5. Surveys with fewer than 20 quadrats
q_count_per_survey = quadrat.groupby('PARENT_KEY').size().reindex(survey['KEY'], fill_value=0)
qc['surveys_lt20_quadrats'] = (q_count_per_survey < 20).sum()

# ── quadrat-level flags ───────────────────────────────────────────────────────
# 6. Quadrats with herbs_present=yes but no species listed
herbs_yes = quadrat['herbs_present'] == 'yes'
no_species = quadrat['herb_species-selected_herb_species_uuids'].isna()
qc['herb_yes_no_species'] = (herbs_yes & no_species).sum()

# 7. Quadrats with herbs_present=no but species listed
herbs_no   = quadrat['herbs_present'] == 'no'
has_species = quadrat['herb_species-selected_herb_species_uuids'].notna()
qc['herb_no_has_species'] = (herbs_no & has_species).sum()

# 8. Missing quadrat GPS
qc['missing_quadrat_gps'] = quadrat['location_quadrat-Latitude'].isna().sum()

# 9. Quadrats missing image
qc['missing_quadrat_image'] = quadrat['quadrat_image'].isna().sum()

# ── additional species flags ──────────────────────────────────────────────────
# 10. Unresolved unknown species (still herb_xxx / wood_xxx)
unk_pattern = addsp['validated_name'].str.match(r'^(herb|wood)_\d+', na=False)
qc['unresolved_unknowns'] = unk_pattern.sum()

print('QA/QC flag summary:')
for flag, count in qc.items():
    print(f'  {flag:<40} : {count}')

In [ ]:
qc_series = pd.Series(qc).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d62728' if v > 0 else '#2ca02c' for v in qc_series]
bars = ax.barh(qc_series.index, qc_series.values, color=colors)
ax.bar_label(bars, padding=3)
ax.set_xlabel('Number of flagged records')
ax.set_title('QA/QC flags — vegetation surveys')
ax.axvline(0, color='black', lw=0.7)
plt.tight_layout()
plt.show()

## 2.12 · Sampling coverage — plots surveyed vs registered vs planned

In [ ]:
n_planned_primary = (centroids['sample_status']=='primary').sum()
n_registered      = vegplots['plot_name'].nunique()
n_surveyed        = survey['plot_selection-plot_name'].nunique()

print('Sampling coverage overview:')
print(f'  Planned plots (primary centroids) : {n_planned_primary}')
print(f'  Registered vegplots               : {n_registered}')
print(f'  Plots with ≥1 herb survey         : {n_surveyed}')
print(f'  Registration coverage             : {n_registered/n_planned_primary*100:.0f}%')
print(f'  Survey coverage (of registered)   : {n_surveyed/n_registered*100:.0f}%')

# Which registered plots are missing surveys?
surveyed_plots = set(survey['plot_selection-plot_name'])
registered_plots = set(vegplots['plot_name'])
not_surveyed = registered_plots - surveyed_plots
print(f'\nRegistered but not yet surveyed ({len(not_surveyed)}): {sorted(not_surveyed)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# funnel chart
stages = ['Planned\n(primary)', 'Registered', 'Surveyed']
counts = [n_planned_primary, n_registered, n_surveyed]
bars = axes[0].barh(stages, counts, color=PALETTE[:3])
axes[0].bar_label(bars, padding=4)
axes[0].set_xlabel('Number of plots')
axes[0].set_title('Sampling pipeline coverage')

# survey count per plot (deduplication view)
surveys_per_plot = survey['plot_selection-plot_name'].value_counts()
surveys_per_plot.value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color=PALETTE[3], width=0.6
)
axes[1].set_xlabel('Number of surveys')
axes[1].set_ylabel('Plots')
axes[1].set_title('How many times was each plot surveyed?')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('Vegetation — sampling coverage', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2.13 · Correlation — canopy height, species richness, GPS accuracy

In [ ]:
# Join plot metadata with richness
regplots_slim = regplots[[
    'plot_selection-plot_name',
    'plot_metadata-canopy_height_m',
    'plot_metadata-canopy_structure',
    'plot_metadata-ecosystem_type_name',
    'plot_metadata-soil_type',
]].rename(columns={
    'plot_selection-plot_name': 'plot_name',
    'plot_metadata-canopy_height_m': 'canopy_height_m',
    'plot_metadata-canopy_structure': 'canopy_structure',
    'plot_metadata-ecosystem_type_name': 'ecosystem_type',
    'plot_metadata-soil_type': 'soil_type',
})

plot_richness_reset = plot_richness.reset_index().rename(columns={'plot_selection-plot_name': 'plot_name'})
combined = plot_richness_reset.merge(regplots_slim, on='plot_name', how='left')

# numeric correlation
num_cols = ['mean_per_q', 'max_per_q', 'canopy_height_m']
corr = combined[num_cols].corr()
print('Correlation matrix (richness × canopy height):')
print(corr.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# scatter: canopy height vs mean species
axes[0].scatter(combined['canopy_height_m'], combined['mean_per_q'],
                color=PALETTE[0], s=60, edgecolors='grey', alpha=0.8)
for _, row in combined.iterrows():
    axes[0].annotate(row['plot_name'].replace('SavMon_LW_', ''),
                     (row['canopy_height_m'], row['mean_per_q']),
                     fontsize=6.5, ha='left', va='bottom')
axes[0].set_xlabel('Canopy height (m)')
axes[0].set_ylabel('Mean species per quadrat')
axes[0].set_title('Canopy height vs herbaceous richness')

# boxplot: species richness by soil type
if combined['soil_type'].notna().sum() > 0:
    sns.boxplot(data=combined, x='soil_type', y='mean_per_q', ax=axes[1], palette='Set2')
    axes[1].set_xlabel('Soil type')
    axes[1].set_ylabel('Mean species per quadrat')
    axes[1].set_title('Species richness by soil type')
    axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('Vegetation — richness vs environmental predictors', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2.14 · Vegetation EDA — key findings summary

| Finding | Detail |
|---------|--------|
| Survey submissions | 32 total; duplicates present for some plots |
| Quadrat records | 640 (20 per survey, as expected) |
| Additional species records | 1,296 across multiple entry modes |
| Completeness — survey | GPS missing for 3 surveys; ReviewState missing for 30 |
| Completeness — quadrat | Species list missing for 146 quadrats (no herbs or all-unknown) |
| Join integrity | All quadrat PARENT_KEYs match a survey KEY; addsp keys match quadrat keys |
| Duplicate surveys | 1+ plot has multiple submissions — review state used to resolve |
| Species list | 370 species (herbaceous + woody); no vernacular names populated |
| species_extra | 250 records (unknown/missing entries); 109 pending review |
| Spatial coverage | 31 primary plots registered; all within compact Lewa area |
| Critical QC flags | herb_yes_no_species; rejected submissions; missing GPS |

> The data pipeline is **survey → quadrat (PARENT_KEY) → additional_species (PARENT_KEY)**. Two-level nesting is clean with no orphaned children. Main quality issues are the known `species_extra` unknowns awaiting expert review.

---
## End of EDA notebook

All cells are self-contained and run top-to-bottom from the project root (`NS/`). Outputs from `bird_challenge.py` and `vegetation_challenge.py` are written to `outputs/` and are independent of this notebook.